# Anveshak: Open-World Remote Sensing Object Detector
### Techfest 2026-27 | IITB-HRTI Centre of Excellence
---
**Kaggle GPU Runner (Tesla T4)**
Runs uninterrupted in the cloud with automatic background execution.

## 1. Verify GPU Setup

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 2. Clone Codebase from GitHub & Install Dependencies

In [ ]:
import os
%cd /kaggle/working
if not os.path.exists("IITBBB"):
    !git clone https://github.com/Likhitha-talent/IITB-Anveshak.git IITBBB
else:
    %cd IITBBB
    !git pull

%cd /kaggle/working/IITBBB
!pip install -r requirements.txt -q
!pip install gdown -q
print("[✓] Codebase ready in:", os.getcwd())

## 3. Download & Extract Dataset (20 GB) into Fast Scratch Disk (/tmp/data)
We extract to `/tmp/data/` (70+ GB NVMe SSD) to keep `/kaggle/working/` clean for checkpoints and submission files.

In [ ]:
import os
import zipfile
import gdown

!mkdir -p /tmp/data
ZIP_PATH = "/tmp/data/round1.zip"
DATASET_FILE_ID = "11Jq4Vu66-VkErgqXkHEtYNJ2BKecVl4p"
URL = f"https://drive.google.com/uc?id={DATASET_FILE_ID}"

if not os.path.exists("/tmp/data/round1"):
    if not os.path.exists(ZIP_PATH):
        print("[*] Downloading round1.zip from Google Drive...")
        gdown.download(URL, ZIP_PATH, quiet=False, fuzzy=True)
        if not os.path.exists(ZIP_PATH):
            print("[*] Fallback downloading via gdown CLI...")
            !gdown {DATASET_FILE_ID} -O {ZIP_PATH} --fuzzy

    if os.path.exists(ZIP_PATH):
        print("[*] Extracting round1.zip into /tmp/data/...")
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall("/tmp/data/")
    else:
        raise FileNotFoundError(f"Failed to download {ZIP_PATH}. Check Google Drive link!")

print("[✓] Dataset extracted successfully:")
!ls -lh /tmp/data


## 4. Train Detector on Set A (From Scratch - Random Initialization)

In [ ]:
import os
%cd /kaggle/working/IITBBB
os.environ['PYTHONPATH'] = '/kaggle/working/IITBBB'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!python -m src.training.train_set_a \
    --config configs/base_config.yaml \
    --data_dir /tmp/data \
    --output_dir /kaggle/working/checkpoints/

## 5. Continual Few-Shot Learning on Set B

In [ ]:
import os
%cd /kaggle/working/IITBBB
os.environ['PYTHONPATH'] = '/kaggle/working/IITBBB'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!python -m src.training.train_set_b \
    --config configs/base_config.yaml \
    --checkpoint_in /kaggle/working/checkpoints/set_a_best.pth \
    --data_dir /tmp/data \
    --output_checkpoint /kaggle/working/checkpoints/round1_final.pth

## 6. Inference & Conflict Resolution

In [ ]:
import os
%cd /kaggle/working/IITBBB
os.environ['PYTHONPATH'] = '/kaggle/working/IITBBB'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!python -m src.inference.predict \
    --checkpoint /kaggle/working/checkpoints/round1_final.pth \
    --test_images_dir /tmp/data/round1/test/images \
    --output_csv /kaggle/working/predictions.csv

## 7. Validate Predictions

In [ ]:
!python /tmp/data/round1/validate_predictions.py \
    /kaggle/working/predictions.csv \
    --tiles /tmp/data/round1/test/tiles.csv \
    --classes /tmp/data/round1/class_list.csv


## 8. Generate Methodology Presentation (.pptx)


In [ ]:
%cd /kaggle/working/IITBBB
!pip install python-pptx -q
!python -m src.utils.generate_ppt /kaggle/working/anveshak_methodology_presentation.pptx


## 9. Package & Download Complete Submission Deliverables


In [ ]:
%cd /kaggle/working

# Package model checkpoint, verified predictions CSV, and methodology PPTX
!zip -j anveshak_round1_submission.zip \
    /kaggle/working/predictions.csv \
    /kaggle/working/checkpoints/round1_final.pth \
    /kaggle/working/anveshak_methodology_presentation.pptx

from IPython.display import FileLink, display
print("[✓] Click the link below to download your complete Round 1 submission package:")
display(FileLink('anveshak_round1_submission.zip'))
